##Parametrização conforme ambiente de execução.

In [1]:
%%capture
import pandas as pd

# Define the base path to the dataset folder
if ENVIRONMENT == 'COLAB':
  # from google.colab import drive
  # drive.mount('/content/drive')

  base_path = '/content/drive/MyDrive/_Mestrado PUCPR/Datasets/'

  num_layers_to_freeze = 6 # You can adjust this number based on your inspection of layer names
  #model_checkpoint = 'neuralmind/bert-base-portuguese-cased'
  #model_checkpoint = 'neuralmind/bert-large-portuguese-cased'
  model_checkpoint = 'dominguesm/legal-bert-base-cased-ptbr'
  EPOCHS = 12
  APPROACH_EXECUTION = 'step'
  reduced_embedding_dim = 512
  per_device_train_batch_size = 4
  per_device_eval_batch_size = 4
  # Define output directory
  output_dir = './results'
  install_evaluate = False
  learning_rate= 2e-5
elif ENVIRONMENT == 'PUCPR':
  num_layers_to_freeze = 0 # Result get after trade-offs layers freezing BigBird (Trade-off freezing layers in BigBird.ipynb)
  base_path = 'dataset/'
  model_checkpoint = 'neuralmind/bert-large-portuguese-cased'
  EPOCHS = 150
  APPROACH_EXECUTION = 'epoch'
  reduced_embedding_dim = 512
  per_device_train_batch_size = 8
  per_device_eval_batch_size = 8
  learning_rate= 1e-5
  # Define output directory
  output_dir = '/dados_abertos_stj/saved_models/hc-bertimbau'

  # Before execution, check install packages
  !pip install tiktoken
  !pip install sentencepiece
  !pip install evaluate

  install_evaluate = True
else:
  raise Exception("Environment of execution is not recognize!")

NameError: name 'ENVIRONMENT' is not defined

In [2]:
def get_base_path(predction_type: str = 'decision'):
  if (predction_type == 'decision'):
    return base_path
  elif (predction_type == 'time'):
    return base_path + 'time-prediction/'
  else:
    raise Exception("Type of prediction is invalid!")

In [3]:
from datetime import datetime, date

def cast_string_to_date(date_string: str) -> date:
    """
    Casts a string in 'DD/MM/YYYY' format to a datetime.date object.
    Handles nulls, empty strings, and whitespace robustly.
    """
    # 1. Check if it's null, not a string, or a NaN object
    if pd.isna(date_string) or not isinstance(date_string, str):
        return None

    # 2. Clean whitespace and check if empty
    clean_date = date_string.strip()
    if not clean_date:
        return None

    try:
        # 3. Standard conversion for dd/mm/yyyy
        return datetime.strptime(clean_date[0:10], '%d/%m/%Y').date()
    except (ValueError, TypeError):
        return None